# Bitcoin RAF (Retrieval-Augmented Forecasting)

Upgrades the original single-cell LSTM notebook to a Retrieval-Augmented Forecasting architecture:
- **Technical indicators** (RSI, MACD, Bollinger Bands, EMA) + time features + lag returns as inputs
- **GRU/LSTM base forecaster** predicting next-step *return* (not raw price -- more stationary), with proper mini-batch training, gradient clipping, early stopping (the original trained full-batch with no validation loop)
- **FAISS retrieval**: the base model's learned embedding for the current market window is used to look up the most similar historical windows, and what actually happened after them
- **Learned blend head**: a small MLP combines the raw forecast with retrieval statistics, rather than a fixed hand-picked weight
- **Gemini reasoning layer** (RAG-style): retrieved historical analogs + the model's numbers are turned into a plain-language market note

Also fixes two real bugs from the original: the target column (Open) didn't match the scaler used to invert predictions (Close), and a `DataLoader` was created but never actually used for training.

**Not financial advice** -- this is a portfolio ML project. Predicting a few basis points of next-minute return correctly does not imply a profitable trading strategy after fees/slippage; say so if you present this.

Run in Colab (GPU optional -- this model is small enough to train on CPU in a few minutes, but GPU is faster).

## 1. Setup

In [ ]:
!git clone https://github.com/pratyu2h/Summer-projects.git
%cd Summer-projects/bitcoin
!pip install -q ta faiss-cpu kaggle google-genai

## 2. Download data (Kaggle API)

Uses the same 1-minute BTC/USD dataset the original notebook used locally (`mczielinski/bitcoin-historical-data`). You need a Kaggle API token (kaggle.com → Account → Create New API Token).

In [ ]:
from google.colab import files
print("Upload your kaggle.json:")
uploaded = files.upload()

import os
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/kaggle.json
!chmod 600 /root/.kaggle/kaggle.json

!kaggle datasets download -d mczielinski/bitcoin-historical-data -p /content/btc_data --unzip

In [ ]:
import pandas as pd
import glob

csv_path = glob.glob("/content/btc_data/*.csv")[0]
df_raw = pd.read_csv(csv_path)
df_raw = df_raw.rename(columns={c: c.strip() for c in df_raw.columns})
df_raw = df_raw.tail(200_000).reset_index(drop=True)  # last ~140 days of 1-min bars; raise/lower to trade off train time vs. data volume
print(df_raw.shape)
df_raw.head()

## 3. Build features and windows

In [ ]:
from data import build_feature_frame, make_windows, time_split, BTCWindowDataset, ALL_FEATURES
from torch.utils.data import DataLoader

feat_df = build_feature_frame(df_raw, timestamp_col="Timestamp", timestamp_is_unix=True)
windows = make_windows(feat_df, seq_len=60, features=ALL_FEATURES)

train_idx, val_idx, test_idx = time_split(len(windows.X), train_frac=0.7, val_frac=0.15)
print(f"train={train_idx.stop} val={val_idx.stop-train_idx.stop} test={len(windows.X)-val_idx.stop}")

In [ ]:
import numpy as np
from sklearn.preprocessing import StandardScaler

# fit scaler on TRAIN ONLY, apply to all splits -- fitting on the full
# dataset would leak val/test statistics into training
X_train_flat = windows.X[train_idx].reshape(-1, windows.X.shape[-1])
scaler = StandardScaler().fit(X_train_flat)

def scale(X):
    shape = X.shape
    return scaler.transform(X.reshape(-1, shape[-1])).reshape(shape).astype(np.float32)

X_train, X_val, X_test = scale(windows.X[train_idx]), scale(windows.X[val_idx]), scale(windows.X[test_idx])
y_train, y_val, y_test = windows.y[train_idx], windows.y[val_idx], windows.y[test_idx]
ts_train = windows.timestamps[train_idx].reset_index(drop=True)

train_ds = BTCWindowDataset(X_train, y_train)
val_ds = BTCWindowDataset(X_val, y_val)
test_ds = BTCWindowDataset(X_test, y_test)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
train_loader_noshuffle = DataLoader(train_ds, batch_size=64, shuffle=False)  # for building the retrieval index -- order must match ts_train
val_loader = DataLoader(val_ds, batch_size=64, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=64, shuffle=False)

## 4. Train the base forecaster

In [ ]:
import torch
from model import BTCForecaster

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

model = BTCForecaster(num_features=len(ALL_FEATURES), hidden_size=128, num_layers=2, cell_type="gru", dropout=0.2)

from train import train_base_model
model, history = train_base_model(model, train_loader, val_loader, device, epochs=30, lr=1e-3, patience=5)

In [ ]:
import matplotlib.pyplot as plt

epochs_r = [h["epoch"] for h in history]
plt.plot(epochs_r, [h["train_loss"] for h in history], label="train")
plt.plot(epochs_r, [h["val_loss"] for h in history], label="val")
plt.title("Huber loss (predicting next-step return)")
plt.xlabel("epoch"); plt.legend(); plt.show()

## 5. Build the FAISS retrieval index

In [ ]:
from retrieval import build_retriever

retriever = build_retriever(model, train_loader_noshuffle, ts_train, device)
print(f"Indexed {retriever.index.ntotal} historical windows, embedding dim {retriever.dim}")

## 6. Train the blend head, and compare against raw-only

In [ ]:
from train import compute_retrieval_features, train_blend_head, rmse

tr_raw, tr_rmean, tr_rstd, tr_y = compute_retrieval_features(model, train_loader_noshuffle, retriever, device)
va_raw, va_rmean, va_rstd, va_y = compute_retrieval_features(model, val_loader, retriever, device)
te_raw, te_rmean, te_rstd, te_y = compute_retrieval_features(model, test_loader, retriever, device)

blend = train_blend_head(tr_raw, tr_rmean, tr_rstd, tr_y, va_raw, va_rmean, va_rstd, va_y, device, epochs=200)

with torch.no_grad():
    blended_test_pred = blend(
        torch.from_numpy(te_raw).float().to(device),
        torch.from_numpy(te_rmean).float().to(device),
        torch.from_numpy(te_rstd).float().to(device),
    ).cpu().numpy()

print(f"Test RMSE, raw forecast only:      {rmse(te_raw, te_y):.6f}")
print(f"Test RMSE, retrieval-blended:      {rmse(blended_test_pred, te_y):.6f}")
print(f"Naive baseline (predict 0 return): {rmse(np.zeros_like(te_y), te_y):.6f}")

In [ ]:
plt.figure(figsize=(14,5))
n_show = 300
plt.plot(te_y[:n_show], label="actual return", alpha=0.7)
plt.plot(te_raw[:n_show], label="raw forecast", alpha=0.7)
plt.plot(blended_test_pred[:n_show], label="retrieval-blended", alpha=0.7)
plt.legend(); plt.title("Next-step return: actual vs. predicted (test set)"); plt.show()

## 7. Gemini reasoning layer

Store your key in Colab Secrets as `GEMINI_API_KEY`. Picks one test-set window and generates a plain-language note grounded in the retrieved historical analogs (RAG pattern).

In [ ]:
from google.colab import userdata
import os
os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

from gemini_reasoning import get_market_commentary

idx = 0
sample_X = torch.from_numpy(X_test[idx:idx+1]).float().to(device)
with torch.no_grad():
    emb = model.encode(sample_X).cpu().numpy()[0]
    raw_pred = float(model.head(model.encode(sample_X)).squeeze(-1).cpu().numpy()[0])

neighbors = retriever.query(emb, k=5)
w_mean, std = retriever.query_batch_stats(emb.reshape(1, -1), k=5)
with torch.no_grad():
    blended_pred = float(blend(
        torch.tensor([raw_pred]).float().to(device),
        torch.from_numpy(w_mean).float().to(device),
        torch.from_numpy(std).float().to(device),
    ).cpu().numpy()[0])

current_indicators = {
    "rsi": float(feat_df["rsi"].iloc[test_idx.start + idx + 59]),
    "macd": float(feat_df["macd"].iloc[test_idx.start + idx + 59]),
    "ema_20": float(feat_df["ema_20"].iloc[test_idx.start + idx + 59]),
}

note = get_market_commentary(current_indicators, raw_pred, blended_pred, neighbors)
print(note)

## 8. Push results back

In [ ]:
import json
with open("training_history.json", "w") as f:
    json.dump({"base_model_history": history,
               "test_rmse_raw": rmse(te_raw, te_y),
               "test_rmse_blended": rmse(blended_test_pred, te_y)}, f, indent=2)

!git config --global user.email "you@example.com"
!git config --global user.name "your-name"
!git add training_history.json
!git commit -m "Add training history from Colab run"
!git push